In [ ]:
import torch
import torch.nn as nn
from amr.dataloaders.dataloader import *
from amr.utils import *
import os


def main(cfgs):

    # 记录日志：PyTorch版本信息
    logger.info('=> PyTorch Version: {}'.format(torch.__version__))

    # 初始化运行环境
    # 设置随机种子保证结果可复现
    # device：根据配置选择CPU或GPU设备
    # pin_memory：用于加速GPU数据传输
    device, pin_memory = init_device(cfgs.seed, cfgs.cpu, cfgs.gpu)
    print(device, pin_memory)

    # 加载数据
    # dataset: 数据集名称
    # Xmode: 数据输入模式
    # batch_size: 批处理大小
    # num_workers: 数据加载线程数
    # mod_type: 调制类型列表
    # 返回: 训练/验证/测试数据加载器、信噪比列表、调制类型列表
    train_loader, valid_loader, test_loader, snrs, mods = AMRDataLoader(
        dataset=cfgs.dataset,
        Xmode=cfgs.params["Xmode"][0],
        batch_size=cfgs.params["batch_size"],
        num_workers=cfgs.workers,
        pin_memory=pin_memory,
        mod_type=cfgs.mod_type)()

    # 初始化模型
    model = init_model(cfgs, network=cfgs.params["network"])
    model.to(device)

    # 初始化损失函数
    criterion = init_loss(cfgs.params["loss"])

    # ------训练模式------
    if cfgs.train:

        # 实例化优化器
        # AdamW优化器带有权重衰减正则化
        # lr: 学习率
        # weight_decay: 权重衰减系数
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=float(cfgs.params["lr"]),
            weight_decay=cfgs.params["weight_decay"])

        # 实例化训练器
        trainer = Trainer(
            model=model,
            device=device,
            optimizer=optimizer,
            lr_decay=cfgs.params["lr_decay"],
            criterion=criterion,
            save_path='results/' + cfgs.method + '/' + cfgs.params["network"] + '/' + cfgs.dataset + '/checkpoints',
            early_stop=cfgs.params["early_stop"])

        # 执行训练循环
        # epochs: 训练轮数
        # 返回训练和验证的损失/准确率历史数据
        train_loss, train_acc, valid_loss, valid_acc = trainer.loop(cfgs.params["epochs"], train_loader, valid_loader)

        # 绘制训练过程曲线
        # 保存训练和验证的损失/准确率变化图
        draw_train(
            train_loss,
            train_acc,
            valid_loss,
            valid_acc,
            save_path='./results/' + cfgs.method + '/' + cfgs.params["network"] + '/' + cfgs.dataset + '/draws')

    # ------测试模式------
    cfgs.train = False
    # 重新加载模型(目前的最佳模型)
    model = init_model(cfgs, network=cfgs.params["network"])
    # 切换到测试模式(禁用dropout等训练特定操作)
    model.to(device)

    # 执行测试
    # test_loss：返回测试损失
    # test_acc：总体准确率
    # test_conf：返回总体混淆矩阵
    # test_conf_snr：返回不同信噪比下的混淆矩阵
    # test_acc_snr：返回不同信噪比下的准确率
    test_loss, test_acc, test_conf, test_conf_snr, test_acc_snr = Tester(model=model,
                                                                         device=device,
                                                                         criterion=criterion,
                                                                         classes=len(cfgs.mod_type),
                                                                         snrs=snrs)(test_loader)
    # 绘制：总体混淆矩阵
    # 可视化所有信噪比下的分类性能
    draw_conf(test_conf,
              save_path='./results/' + cfgs.method + '/' + cfgs.params["network"] + '/' + cfgs.dataset + '/draws',
              labels=mods,
              order="total")

    # 绘制：不同信噪比下的混淆矩阵
    for i in range(len(snrs)):

        # 记录日志：各个信噪比下的测试结果
        logger.info(f'test_snr : {snrs[i]:.0f} | '
                    f'test_acc : {test_acc_snr[i]:.4f}')

        draw_conf(test_conf_snr[i],
                  save_path='./results/' + cfgs.method + '/' + cfgs.params["network"] + '/' + cfgs.dataset + '/draws',
                  labels=mods,
                  order=str(snrs[i]))

    # 绘制：准确率-信噪比曲线
    # 评估模型在不同噪声环境下的鲁棒性
    draw_acc(snrs, test_acc_snr,
             save_path='./results/' + cfgs.method + '/' + cfgs.params["network"] + '/' + cfgs.dataset + '/draws')

    # 记录日志：最终测试结果
    logger.info(f'test_loss : {test_loss:.4e} | '
                f'test_acc : {test_acc:.4f}')


if __name__ == '__main__':
    cfgs = get_cfgs()
    main(cfgs)